# LLM Skill Analysis

In [1]:
import json
import pandas as pd
import ollama
from pathlib import Path
import numpy as np

All csv files being written to 'analysis' folder

Uses ollama model 
1. pip install ollama
2. ollama run llama3


In [4]:
SUBSET_PATH = Path("llm_analysis_subset.csv")
BASELINES_PATH = Path("full_alexa_skills_with_baselines.csv")
DIRECTORY = Path("analysis")

MODEL = "llama3"

available = [m.model for m in ollama.list().models]
if any(MODEL in m for m in available):
    print("Using:", MODEL)
else:
    print(MODEL, "Not Found ERROR")

Using: llama3


## 1. Load Subset

In [5]:
# llm_analysis_subset.csv
subset = pd.read_csv(SUBSET_PATH)
print("Subset:", subset.shape)

# Alexa dataset
baselines = pd.read_csv(BASELINES_PATH)
print("Baselines:", baselines.shape)

subset[["Skill Name", "Invocation Name", "Category", "baseline_risk_score", "selection_reasons"]].head()

Subset: (1244, 62)
Baselines: (42202, 61)


,Skill Name,Invocation Name,Category,baseline_risk_score,selection_reasons
0,Cricket Facts,cricket facts,Business & Finance,0.834719,baseline_high_risk;high_genericity_ambiguity
1,bitcoin facts,crypto facts,Business & Finance,0.743463,baseline_high_risk;high_genericity_ambiguity
2,Chore chart,chore chart,Business & Finance,0.656681,baseline_high_risk
3,AI NVR Integration Test,my assistant,Business & Finance,0.643772,baseline_high_risk
4,Cryptocurrency,crypto currency,Business & Finance,0.639859,baseline_high_risk


## 2. Prompt 

In [6]:
def build_prompt(row):
    return f"""
You are analyzing potential security and usability risks of Alexa skill invocation names.

Evaluate the following skill and return a structured JSON response.

Skill Information:
- Skill Name: {row['Skill Name']}
- Invocation Name: {row['Invocation Name']}
- Category: {row['Category']}
- Description: {row['Description']}
- Average Rating: {row['Average Rating']}
- Number of Ratings: {row['Number of Rating']}

Context:
- Invocation names are spoken by users
- Users may mispronounce, abbreviate, or paraphrase.
- Attackers may create similar-sounding or similar-looking names.

Evaluate the following dimensions (0-5 scale):
1. Genericity Risk (how generic/common the name is)
2. Ambiguity Risk (multiple plausible meanings or interpretations)
3. Semantic Confusion Risk (could be confused with other skills)
4. Squatting Risk (ease of imitation or malicious variants)
5. Overall Risk (holistic judgment)

Return ONLY valid JSON in this format:

{{
    "genericity_risk": int,
    "ambiguity_risk": int,
    "semantic_confusion_risk": int,
    "squatting_risk": int,
    "overall_risk": int,
    "reasoning": "concise explanation (2-4 sentences)"
}}
"""

In [7]:
def build_full_prompt(row):
    return f"""
You are analyzing potential security and usability risks of Alexa skill invocation names.

Evaluate the following skill and return a structured JSON response.

Skill Information:
- Skill Name: {row['Skill Name']}
- Invocation Name: {row['Invocation Name']}
- Category: {row['Category']}
- Description: {row['Description']}
- Average Rating: {row['Average Rating']}
- Number of Ratings: {row['Number of Rating']}

Context:
- Invocation names are spoken aloud by users.
- Users may mispronounce, abbreviate, or paraphrase names.
- Attackers may create similar sounding or semantically similar invocation names.

Assign an integer score from 0 to 5 for each dimension.

Use the FULL scale:
0 = no meaningful risk
1 = very low risk
2 = low risk
3 = moderate risk
4 = high risk
5 = very high risk

Do not default to middle scores.
Use 0 or 5 when appropriate.

Scoring dimensions:

1. Genericity Risk
0 = highly distinctive and unique
1 = mostly distinctive
2 = somewhat common
3 = moderately generic
4 = highly generic/common
5 = extremely generic, broad everyday phrase

2. Ambiguity Risk
0 = only one clear meaning
1 = very little ambiguity
2 = minor ambiguity
3 = moderate ambiguity
4 = multiple plausible meanings
5 = highly ambiguous, many interpretations

3. Semantic Confusion Risk
0 = unlikely to be confused with other skills
1 = minimal confusion potential
2 = slight confusion potential
3 = moderate confusion potential
4 = high confusion potential
5 = extremely easy to confuse with other skills

4. Squatting Risk
0 = difficult to imitate or exploit
1 = very low squatting potential
2 = low squatting potential
3 = moderate squatting potential
4 = high squatting potential
5 = extremely easy to imitate with malicious variants

5. Overall Risk
0 = negligible overall risk
1 = very low overall risk
2 = low overall risk
3 = moderate overall risk
4 = high overall risk
5 = severe overall risk

Return ONLY valid JSON in this format:

{{
    "genericity_risk": int,
    "ambiguity_risk": int,
    "semantic_confusion_risk": int,
    "squatting_risk": int,
    "overall_risk": int,
    "reasoning": "concise explanation (2-4 sentences)"
}}
"""

## 3. Load Functions

In [8]:
SCORE_FIELDS = ["genericity_risk", "ambiguity_risk", "semantic_confusion_risk",
                "squatting_risk", "overall_risk"]

def run_model(prompt):
    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    return resp["message"]["content"].strip()


def parse_response(resp):
    base = {}

    for field in SCORE_FIELDS:
        base["llm_" + field] = None


    base["llm_reasoning"] = None
    if resp is None:
        return base

    start = resp.find("{")
    end = resp.rfind("}")

    if start == -1 or end == -1:
        return base

    try:
        json_str = resp[start:end + 1].replace("\\'", "'")
        obj = json.loads(json_str)


        for field in SCORE_FIELDS:
            if field in obj:
                base["llm_" + field] = int(obj[field])
        base["llm_reasoning"] = str(obj.get("reasoning", ""))
    except (json.JSONDecodeError, KeyError, ValueError):
        pass

    return base

## 4. Run Analysis

In [9]:
new_rows = []

for index, row in subset.iterrows():
    raw_response = run_model(build_full_prompt(row))
    parsed_data  = parse_response(raw_response)

    combined_row = row.to_dict()
    combined_row.update(parsed_data)
    new_rows.append(combined_row)

    if (index + 1) % 25 == 0:
        print(index + 1, "rows completed")

results_df = pd.DataFrame(new_rows)
print("Results:", results_df.shape)

25 rows completed
50 rows completed
75 rows completed
100 rows completed
125 rows completed
150 rows completed
175 rows completed
200 rows completed
225 rows completed
250 rows completed
275 rows completed
300 rows completed
325 rows completed
350 rows completed
375 rows completed
400 rows completed
425 rows completed
450 rows completed
475 rows completed
500 rows completed
525 rows completed
550 rows completed
575 rows completed
600 rows completed
625 rows completed
650 rows completed
675 rows completed
700 rows completed
725 rows completed
750 rows completed
775 rows completed
800 rows completed
825 rows completed
850 rows completed
875 rows completed
900 rows completed
925 rows completed
950 rows completed
975 rows completed
1000 rows completed
1025 rows completed
1050 rows completed
1075 rows completed
1100 rows completed
1125 rows completed
1150 rows completed
1175 rows completed
1200 rows completed
1225 rows completed
Results: (1244, 68)


In [10]:
results_df.to_csv(DIRECTORY/"llm_results.csv")
print("saved")

saved


## 5. Score Distributions

In [41]:
llm_columns = []

for field in SCORE_FIELDS:
    column_name = "llm_"+str(field) 
    llm_columns.append(column_name)  

new_data = results_df.dropna(subset=llm_columns).copy()
description = new_data[llm_columns].describe().round(2)

## 6. Compare LLM vs Baseline

In [70]:
new_data["llm_composite"] = new_data["llm_overall_risk"]

corr = new_data[["baseline_risk_score", "llm_composite"]].corr().iloc[0, 1]
print(f"baseline_risk_score vs llm_overall_risk: {corr:.5f}")

pop_quartiles = baselines["baseline_risk_score"].quantile([0.25, 0.5, 0.75])
new_data["baseline_population_tier"] = pd.cut( new_data["baseline_risk_score"],
    bins=[-np.inf, pop_quartiles[0.25], pop_quartiles[0.5], pop_quartiles[0.75], np.inf],
    labels=["Q1 low", "Q2", "Q3", "Q4 high"],
)

new_data["llm_quartile"] = pd.qcut(new_data["llm_composite"], q=4, labels=False, duplicates="drop")

print()
print("LLM score distribution:")
print(new_data["llm_composite"].value_counts().sort_index())


print()
print("Baseline vs LLM overall risk:")
print(pd.crosstab(new_data["baseline_population_tier"], new_data["llm_composite"]))

comparison_cols = [
    "Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "baseline_population_tier",
    "llm_genericity_risk", "llm_ambiguity_risk",
    "llm_semantic_confusion_risk", "llm_squatting_risk",
    "llm_overall_risk", "llm_reasoning",
]
new_data[comparison_cols].to_csv(DIRECTORY / "llm_vs_baseline.csv", index=False)
print()
print("Wrote")

baseline_risk_score vs llm_overall_risk: 0.00843

LLM score distribution:
llm_composite
1    252
2    759
3    104
4     75
5     54
Name: count, dtype: int64

Baseline vs LLM overall risk:
llm_composite              1    2   3   4   5
baseline_population_tier                     
Q1 low                    81  141  35  20  13
Q2                        44  112  22  11   9
Q3                        38  114  22  12   3
Q4 high                   89  392  25  32  29

Wrote


## 7. False positives vs False Negatives

In [71]:
fp = new_data[
    (new_data["baseline_risk_score"] >= new_data["baseline_risk_score"].quantile(0.75)) &
    (new_data["llm_composite"] <=new_data["llm_composite"].quantile(0.25))
].sort_values("baseline_risk_score", ascending=False)

print(f"False Positives: {len(fp)} skills")
fp[["Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "llm_composite", "llm_reasoning"]].head(10)

False Positives: 270 skills


,Skill Name,Invocation Name,Category,baseline_risk_score,llm_composite,llm_reasoning
994,cricket facts,cricket facts,Sports,0.834719,1,The invocation name 'cricket facts' is highly ...
993,Cricket Facts,cricket facts,Sports,0.834719,2,The invocation name 'cricket facts' is mostly ...
992,Cricket Facts,cricket facts,Sports,0.834719,2,The invocation name 'cricket facts' is mostly ...
991,Cricket Facts,cricket facts,Sports,0.834719,2,The invocation name 'cricket facts' is mostly ...
990,Cricket Facts,cricket facts,Sports,0.834719,2,The invocation name 'cricket facts' is mostly ...
989,Cricket Facts,cricket facts,Sports,0.834719,1,The invocation name 'cricket facts' is highly ...
988,Amazing Cricket Facts,cricket facts,Sports,0.834719,2,The invocation name 'cricket facts' is mostly ...
987,Cricket Facts,cricket facts,Sports,0.834719,1,The invocation name 'cricket facts' is highly ...
286,Cricket_Facts,cricket facts,Game and Trivia,0.834719,2,The invocation name 'cricket facts' is mostly ...
186,Tykoon's cricket facts,cricket facts,Education & Reference,0.834719,2,The invocation name 'cricket facts' is mostly ...


In [72]:
fn = new_data[
    (new_data["baseline_risk_score"] <= new_data["baseline_risk_score"].quantile(0.25)) &
    (new_data["llm_composite"] >=new_data["llm_composite"].quantile(0.75))
].sort_values("llm_composite", ascending=False)

print(f"False Negatives: {len(fn)} skills")
fn[["Skill Name", "Invocation Name", "Category",
    "baseline_risk_score", "llm_composite", "llm_reasoning"]].head(10)

disagreement_cols = ["Skill Name", "Invocation Name", "Category",
                     "baseline_risk_score", "llm_composite", "llm_reasoning"]

fp[disagreement_cols].to_csv(DIRECTORY/"false_positives.csv", index=False)
fn[disagreement_cols].to_csv(DIRECTORY/"false_negatives.csv", index=False)

print()
print(f"Wrote analysis/false_positives.csv {len(fp)} rows")
print(f"Wrote analysis/false_negatives.csv {len(fn)} rows")

False Negatives: 225 skills

Wrote analysis/false_positives.csv 270 rows
Wrote analysis/false_negatives.csv 225 rows


In [68]:
import pandas as pd
from pathlib import Path

DIRECTORY = Path("analysis")

fp = pd.read_csv(DIRECTORY / "false_positives.csv")
fn = pd.read_csv(DIRECTORY / "false_negatives.csv")
new_data = pd.read_csv(DIRECTORY / "llm_results.csv")

In [73]:
fp_by_category = (
    fp.groupby("Category")
      .agg(
          fp_count=("Skill Name", "count"),
          avg_baseline=("baseline_risk_score", "mean"),
          avg_llm=("llm_composite", "mean")
      )
      .sort_values("fp_count", ascending=False)
)

fp_by_category

,fp_count,avg_baseline,avg_llm
Category,,,
Novelty and Humor,29,0.762009,1.620690
Game and Trivia,28,0.788316,1.750000
Education & Reference,27,0.801487,1.888889
Kids,27,0.776496,1.925926
Lifestyle,23,0.725585,1.913043
Sports,21,0.757208,1.809524
Movie and TV,19,0.740907,1.842105
Health and Fitness,17,0.700579,1.764706
productivity,17,0.725669,1.941176


In [74]:
fn_by_category = (
    fn.groupby("Category")
      .agg(
          fn_count=("Skill Name", "count"),
          avg_baseline=("baseline_risk_score", "mean"),
          avg_llm=("llm_composite", "mean")
      )
      .sort_values("fn_count", ascending=False)
)

fn_by_category

,fn_count,avg_baseline,avg_llm
Category,,,
Communication,21,0.230058,2.714286
Shopping,19,0.230368,2.315789
Smart Home,17,0.206276,2.470588
Business & Finance,15,0.229644,2.133333
Utilities,12,0.239721,2.083333
Game and Trivia,12,0.214743,3.000000
Music and Audio,11,0.240607,2.727273
Social,10,0.215438,3.500000
Home services,10,0.234484,2.600000


In [75]:
category_sizes = new_data.groupby("Category").size().rename("total_skills")

In [76]:
fp_rate = fp_by_category.join(category_sizes)
fp_rate["fp_rate"] = fp_rate["fp_count"] / fp_rate["total_skills"]
fp_rate.sort_values("fp_rate", ascending=False)

,fp_count,avg_baseline,avg_llm,total_skills,fp_rate
Category,,,,,
Kids,27,0.776496,1.925926,52,0.519231
Novelty and Humor,29,0.762009,1.620690,56,0.517857
Game and Trivia,28,0.788316,1.750000,56,0.500000
Education & Reference,27,0.801487,1.888889,56,0.482143
Lifestyle,23,0.725585,1.913043,48,0.479167
Sports,21,0.757208,1.809524,46,0.456522
Movie and TV,19,0.740907,1.842105,46,0.413043
productivity,17,0.725669,1.941176,50,0.340000
Food and Drink,14,0.737835,1.857143,46,0.304348


In [77]:
fn_rate = fn_by_category.join(category_sizes)
fn_rate["fn_rate"] = fn_rate["fn_count"] / fn_rate["total_skills"]
fn_rate.sort_values("fn_rate", ascending=False)

,fn_count,avg_baseline,avg_llm,total_skills,fn_rate
Category,,,,,
Communication,21,0.230058,2.714286,67,0.313433
Shopping,19,0.230368,2.315789,63,0.301587
Smart Home,17,0.206276,2.470588,65,0.261538
Business & Finance,15,0.229644,2.133333,68,0.220588
Game and Trivia,12,0.214743,3.000000,56,0.214286
Music and Audio,11,0.240607,2.727273,54,0.203704
Utilities,12,0.239721,2.083333,59,0.203390
Social,10,0.215438,3.500000,50,0.200000
productivity,10,0.236465,2.800000,50,0.200000


## 8. Category Summary

In [78]:
summary = (
    
    new_data.groupby("Category").agg(
        skill_count = ("Skill Name","count"),
        mean_baseline = ("baseline_risk_score","mean"),
        mean_llm_genericity_risk = ("llm_genericity_risk","mean"),
        mean_llm_ambiguity_risk = ("llm_ambiguity_risk","mean"),
        mean_llm_semantic_confusion = ("llm_semantic_confusion_risk","mean"),
        mean_llm_squatting_risk = ("llm_squatting_risk","mean"),
        mean_llm_overall_risk = ("llm_overall_risk","mean"),
        mean_llm_composite = ("llm_composite","mean"),
    ).sort_values("mean_llm_overall_risk", ascending=False).reset_index()
)
summary.to_csv(DIRECTORY /"llm_category_summary.csv", index=False)
summary.round(2)

,Category,skill_count,mean_baseline,mean_llm_genericity_risk,mean_llm_ambiguity_risk,mean_llm_semantic_confusion,mean_llm_squatting_risk,mean_llm_overall_risk,mean_llm_composite
0,Communication,67,0.41,2.19,0.97,2.12,2.19,2.69,2.69
1,Social,50,0.49,2.24,0.82,1.94,2.10,2.64,2.64
2,Smart Home,65,0.39,1.92,0.86,1.85,2.03,2.45,2.45
3,Home services,53,0.41,1.68,0.62,1.89,1.85,2.40,2.40
4,productivity,50,0.51,1.94,0.52,1.66,1.74,2.38,2.38
5,Music and Audio,54,0.46,1.52,0.54,1.63,1.63,2.31,2.31
6,Connected_car,49,0.39,1.55,0.57,1.67,1.69,2.31,2.31
7,Shopping,63,0.34,1.84,0.70,1.73,1.75,2.30,2.30
8,Utilities,59,0.40,1.71,0.42,1.41,1.46,2.15,2.15
9,Food and Drink,46,0.49,1.48,0.65,1.52,1.54,2.13,2.13
